In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [ ]:
S3_BUCKET = "datalake-teste"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

In [3]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/01 15:10:01 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 192.168.0.130 instead (on interface enp3s0)
26/09/01 15:10:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-17b90e55-135b-4c50-bc0e-49ba134a7268;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 283ms :: artifacts dl 12ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [6]:
# Lê dados do S3
df_bronze = spark.read.csv(f"{S3_BRONZE}/dados.csv", header=True, inferSchema=True)
df_bronze.show()

26/09/01 15:12:33 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://dados-teste/bronze/dados.csv.
org.apache.hadoop.fs.s3a.UnknownStoreException: `s3a://dados-teste/bronze/dados.csv': getFileStatus on s3a://dados-teste/bronze/dados.csv: com.amazonaws.services.s3.model.AmazonS3Exception: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 0bcd97f7-2a69-4b05-934f-1497afd899d2; S3 Extended Request ID: null; Proxy: null), S3 Extended Request ID: null:NoSuchBucket: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 0bcd97f7-2a69-4b05-934f-1497afd899d2; S3 Extended Request ID: null; Proxy: null)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:263)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFil

Py4JJavaError: An error occurred while calling o52.csv.
: org.apache.hadoop.fs.s3a.UnknownStoreException: `s3a://dados-teste/bronze/dados.csv': getFileStatus on s3a://dados-teste/bronze/dados.csv: com.amazonaws.services.s3.model.AmazonS3Exception: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 8d31a931-c558-4460-81f8-efe8594dc072; S3 Extended Request ID: null; Proxy: null), S3 Extended Request ID: null:NoSuchBucket: The specified bucket does not exist (Service: Amazon S3; Status Code: 404; Error Code: NoSuchBucket; Request ID: 8d31a931-c558-4460-81f8-efe8594dc072; S3 Extended Request ID: null; Proxy: null)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:263)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3858)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:368

In [16]:
# Filtro (valores > 100)
df_filtrado = df_bronze.filter(col('valor') > 100)

print(f"Registros com valor > 100: {df_filtrado.count()}")
df_filtrado.show()

Registros com valor > 100: 5
+---+---------+-----+-----------+
| id|     nome|valor|  categoria|
+---+---------+-----+-----------+
|  2|Produto B|  150|Eletrônicos|
|  4|Produto D|  200|     Livros|
|  5|Produto E|  120|Eletrônicos|
|  7|Produto G|  250|     Livros|
|  9|Produto I|  180|       Casa|
+---+---------+-----+-----------+



In [10]:
df_filtrado.write.mode("overwrite").csv(f"{S3_SILVER}/df_filtrado", header=True)

26/08/31 23:34:52 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/08/31 23:34:52 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [11]:
df_silver = spark.read.csv(f"{S3_SILVER}/df_filtrado", header=True, inferSchema=True)


df_silver.show()

+---+---------+-----+-----------+
| id|     nome|valor|  categoria|
+---+---------+-----+-----------+
|  2|Produto B|  150|Eletrônicos|
|  4|Produto D|  200|     Livros|
|  5|Produto E|  120|Eletrônicos|
|  7|Produto G|  250|     Livros|
|  9|Produto I|  180|       Casa|
+---+---------+-----+-----------+

